# Joggy-PicX — Fine-tune YOLOv8-nano for bib detection

Trains a 1-class bib detector and exports ONNX ready to drop into
`apps/backend/models/yolov8n_bib.onnx`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your Roboflow YOLO-format dataset zip to `/MyDrive/joggy-bib/dataset.zip`
3. Run all cells (Runtime → Run all)

Expected total time: **30–60 minutes** on T4 GPU at default settings.

## 1. Setup — mount Drive + install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pin ultralytics version — must match the runtime in apps/backend production
# (yolov8 export ONNX format is stable across 8.x — keep at 8.3.x)
!pip install -q 'ultralytics>=8.3.0,<9.0' onnx onnxruntime
import ultralytics, torch
print(f'ultralytics={ultralytics.__version__}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')

## 2. Unpack dataset

The Roboflow YOLO zip should contain:
```
data.yaml
train/images/*.jpg
train/labels/*.txt
valid/images/*.jpg
valid/labels/*.txt
(optional) test/images/*.jpg, test/labels/*.txt
```

In [ ]:
import os, shutil, zipfile, pathlib

DRIVE_ZIP = '/content/drive/MyDrive/joggy-bib/dataset.zip'
WORK_DIR = pathlib.Path('/content/dataset')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(WORK_DIR)

# Locate data.yaml — Roboflow may put it nested
for p in WORK_DIR.rglob('data.yaml'):
    DATA_YAML = p
    break
else:
    raise FileNotFoundError('No data.yaml in zip — check Roboflow export format')

print(f'Dataset root: {DATA_YAML.parent}')
print(DATA_YAML.read_text())

In [ ]:
# Quick sanity — count images per split
for split in ('train', 'valid', 'test'):
    p = DATA_YAML.parent / split / 'images'
    n = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png'))) if p.exists() else 0
    print(f'  {split:6s}: {n} images')

## 3. Train YOLOv8-nano (1-class bib)

Hyperparameters:
- **epochs=100** — early stop on val mAP plateau
- **imgsz=640** — standard yolov8n input
- **batch=-1** — auto-fit T4 VRAM
- **patience=20** — early stop if no improvement for 20 epochs

In [ ]:
from ultralytics import YOLO

# Start from COCO-pretrained nano weights — transfer learning is much faster
# than training from scratch on our small dataset.
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=-1,              # auto-fit GPU memory
    patience=20,           # early stop
    project='joggy-bib',
    name='run1',
    exist_ok=True,
    save_period=10,        # save checkpoint every 10 epochs (resilience)
    plots=True,
    verbose=True,
)

## 4. Inspect training metrics

In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

run_dir = pathlib.Path('joggy-bib/run1')
for png in ('results.png', 'PR_curve.png', 'confusion_matrix.png'):
    p = run_dir / png
    if p.exists():
        print(f'--- {png} ---')
        display(IPyImage(str(p)))

In [ ]:
# Print the final val metrics — these go into your release notes
best = YOLO('joggy-bib/run1/weights/best.pt')
metrics = best.val(data=str(DATA_YAML), imgsz=640, plots=False, verbose=False)
print(f'mAP50    = {metrics.box.map50:.4f}')
print(f'mAP50-95 = {metrics.box.map:.4f}')
print(f'Precision= {metrics.box.mp:.4f}')
print(f'Recall   = {metrics.box.mr:.4f}')

## 5. Spot-check predictions on validation images

In [ ]:
import random
valid_imgs = list((DATA_YAML.parent / 'valid' / 'images').glob('*.jpg'))[:8]
predictions = best.predict(valid_imgs, imgsz=640, conf=0.25, save=True, project='joggy-bib', name='preview', exist_ok=True)
for r in predictions[:3]:
    display(IPyImage(r.save_dir + '/' + pathlib.Path(r.path).name))

## 6. Export to ONNX (production format)

Same call as `tools/export/export_yolo.py` — produces tensor shape `[1, 5, 8400]` which is what `apps/backend/joggy/ai/bib_detector.py` expects for a 1-class model.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, simplify=True, opset=17)
print(f'ONNX exported: {onnx_path}')

# Verify with onnxruntime — same engine production uses
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
print(f'  input shape:  {sess.get_inputs()[0].shape}    name: {sess.get_inputs()[0].name}')
print(f'  output shape: {sess.get_outputs()[0].shape}  name: {sess.get_outputs()[0].name}')

## 7. Copy artifacts back to Drive

After this cell finishes, look in `/MyDrive/joggy-bib/output/` on your computer's Drive sync. Download `yolov8n_bib.onnx` and drop it into the repo at `apps/backend/models/`.

In [ ]:
import shutil, pathlib
out_dir = pathlib.Path('/content/drive/MyDrive/joggy-bib/output')
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(onnx_path, out_dir / 'yolov8n_bib.onnx')
shutil.copy('joggy-bib/run1/weights/best.pt', out_dir / 'best.pt')
shutil.copy('joggy-bib/run1/results.png', out_dir / 'results.png')

# Write a small metrics summary
summary = f'''Joggy-PicX bib detector — training summary
==========================================
Dataset: {DATA_YAML}
Run:     joggy-bib/run1

Validation metrics:
  mAP50    = {metrics.box.map50:.4f}
  mAP50-95 = {metrics.box.map:.4f}
  Precision= {metrics.box.mp:.4f}
  Recall   = {metrics.box.mr:.4f}

Files in this folder:
  yolov8n_bib.onnx  → drop into apps/backend/models/
  best.pt           → PyTorch weights (for re-export or further training)
  results.png       → training curves
'''
(out_dir / 'README.txt').write_text(summary)
print(summary)
print(f'Saved to: {out_dir}')

## Done!

Next steps (off Colab):

1. Download `/MyDrive/joggy-bib/output/yolov8n_bib.onnx` to your computer
2. Drop it into `apps/backend/models/yolov8n_bib.onnx`
3. Run holdout evaluation: `python tools/train/eval_bib.py --model apps/backend/models/yolov8n_bib.onnx --images /path/to/20-holdout-images/`
4. If metrics meet success criteria → restart backend worker → live!

If you want to retrain with more data:
- Add more images to Roboflow → generate new version → re-export → re-upload zip → re-run this notebook